# 05 · Обучение CNN из готовых TAR
В Colab выберите среду с GPU. Выполните ячейки по порядку. Восстановление и обучение запускаются отдельными переключателями.

Результаты validation относятся к этому разбиению. REAL и FAKE происходят из разных источников. На оценку могут влиять источник, кодирование и дополнение тишиной. Разделение по дикторам не подтверждено. Несколько фрагментов могут относиться к одной записи, поэтому основные метрики считаются по исходному `sample_id`. Финальная оценка на `test_unseen` выполняется отдельно.

In [ ]:
# 1. Репозиторий и зависимости Colab
import subprocess
import sys
from pathlib import Path

repo = Path('/content/audio-deepfake-detector')
url = 'https://github.com/Unvand/audio-deepfake-detector.git'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab-train.txt')],
    check=True,
)
sys.path.insert(0, str(repo))

In [ ]:
# 2. Google Drive и GPU
import torch
from google.colab import drive, output

drive.mount('/content/drive')
output.enable_custom_widget_manager()
if not torch.cuda.is_available():
    raise RuntimeError('Выберите среду Colab с GPU и повторите запуск этой ячейки.')
print(torch.cuda.get_device_name(0))

root = Path('/content/drive/MyDrive/VKR_RuASD')
source_root = root / 'prepared_v1'
archive_root = root / 'prepared_v1_tar'
cache_root = Path('/content/ruasd_prepared_cache')
training_root = root / 'training_v1'
splits = ('train', 'validation')

In [ ]:
# 3. Проверка готовых TAR и локального кэша
from audio_deepfake_detector.src.colab_cache import (
    check_local_cache,
    require_ready_cache,
)
from audio_deepfake_detector.src.colab_tar_cache import (
    list_ready_tar_archives,
    restore_tar_cache,
)


def cache_status():
    return check_local_cache(
        source_root, cache_root, splits, verify_checksums=True, compare_source_sizes=False,
    )

try:
    archives = list_ready_tar_archives(source_root, archive_root, splits)
    print('TAR на Drive', ', '.join(str(path.relative_to(archive_root)) for path in archives))
except (OSError, ValueError) as error:
    print('Проверка TAR', error)

status = cache_status()
if status.ready:
    print('Кэш готов.', status.valid_files, 'FLAC')
else:
    print(f'Готово {status.valid_files}/{status.expected_files} FLAC. Включите RUN_RESTORE_CACHE.')

In [ ]:
# 4. Синхронное восстановление train и validation
RUN_RESTORE_CACHE = False

if RUN_RESTORE_CACHE:
    status = cache_status()
    if status.ready:
        print('Восстановлено 0, пропущено', status.valid_files, 'ошибок 0')
    else:
        counts = {'copied': 0}
        def record_progress(_done, _total, stats):
            counts.update(stats)

        try:
            result = restore_tar_cache(
                source_root, archive_root, cache_root, splits, progress_callback=record_progress,
            )
            require_ready_cache(source_root, cache_root, splits, compare_source_sizes=False)
        except (OSError, ValueError, RuntimeError):
            print('Восстановлено из завершённых частей', counts['copied'], 'ошибок 1')
            raise
        print(f"Восстановлено {result['restored_files']}, пропущено {result['skipped_files']}, ошибок 0")
    print('Данные готовы:', cache_root)

In [ ]:
# 5. Панель параметров CNN
import ipywidgets as widgets
from IPython.display import display

from audio_deepfake_detector.src.cnn_training import (
    MelConfig,
    TrainingConfig,
    configs_from_mapping,
    inspect_training_run,
    resolve_run_directory,
    train_cnn,
)

defaults = TrainingConfig()
mel_defaults = MelConfig()
controls = {}
for key, label in (
    ('batch_size', 'Batch size'), ('epochs', 'Epochs'),
    ('learning_rate', 'Learning rate'), ('weight_decay', 'Weight decay'),
    ('early_stopping_patience', 'Patience'), ('num_workers', 'Workers'),
    ('seed', 'Seed'), ('balance_classes', 'Balance classes'), ('use_amp', 'Mixed precision AMP'),
):
    value = getattr(defaults, key)
    widget = widgets.Checkbox if isinstance(value, bool) else (
        widgets.IntText if isinstance(value, int) else widgets.FloatText
    )
    controls[key] = widget(value=value, description=label)
run_id = widgets.Text(value='baseline_cached_v1', description='run_id')
resume = widgets.Checkbox(value=True, description='Resume training')

mel_controls = {}
for key, label in (
    ('n_fft', 'FFT'), ('win_length', 'Window'), ('hop_length', 'Hop'),
    ('n_mels', 'Mel bins'), ('f_min', 'f_min'), ('f_max', 'f_max'),
):
    value = getattr(mel_defaults, key)
    widget = widgets.IntText if isinstance(value, int) else widgets.FloatText
    mel_controls[key] = widget(value=value, description=label)

advanced = widgets.Accordion(children=[widgets.VBox([
    widgets.IntText(value=mel_defaults.sample_rate, description='Sample rate', disabled=True),
    *mel_controls.values(),
])], selected_index=None)
advanced.set_title(0, 'Дополнительные настройки')
display(widgets.VBox([run_id, resume, *controls.values(), advanced]))

def current_run():
    values = {key: widget.value for key, widget in (controls | mel_controls).items()}
    mel, training = configs_from_mapping(values)
    return resolve_run_directory(training_root, run_id.value.strip()), mel, training, resume.value

In [ ]:
# 6. Checkpoint и доступность Resume
try:
    run_dir, mel, training, do_resume = current_run()
    next_epoch = inspect_training_run(run_dir, mel, training, resume=do_resume)
    print('Каталог эксперимента', run_dir)
    if next_epoch > training.epochs:
        print('Все запрошенные эпохи уже завершены.')
    else:
        print(f'Следующая эпоха {next_epoch}/{training.epochs}')
except (OSError, ValueError) as error:
    print(error)

In [ ]:
# 7. Открыть TensorBoard
import shlex

from tensorboard import notebook as tensorboard_notebook

events = resolve_run_directory(training_root, run_id.value.strip()) / 'tensorboard'
if events.is_dir():
    tensorboard_notebook.start(f'--logdir {shlex.quote(str(events))}')
else:
    print('События появятся после запуска обучения. Затем выполните эту ячейку снова.')

In [ ]:
# 8. Синхронное обучение CNN
RUN_TRAINING = False

if RUN_TRAINING:
    run_dir, mel, training, do_resume = current_run()
    next_epoch = inspect_training_run(run_dir, mel, training, resume=do_resume)
    require_ready_cache(source_root, cache_root, splits, compare_source_sizes=False)
    print(f'Следующая эпоха {next_epoch}/{training.epochs}. Результаты сохраняются в {run_dir}')
    _, history = train_cnn(cache_root, run_dir, mel, training, resume=do_resume)
    if history:
        last = history[-1]
        best = max(history, key=lambda row: row['validation_source']['f1'])
        print(f"Последняя эпоха {last['epoch'] + 1}, лучшая {best['epoch'] + 1}. "
              f"Лучший validation F1 по sample_id {best['validation_source']['f1']:.4f}.")
        print('Checkpoint', run_dir / 'last_checkpoint.pth')

In [ ]:
# 9. История обучения и результаты
import json

import matplotlib.pyplot as plt
import pandas as pd

history_path = resolve_run_directory(training_root, run_id.value.strip()) / 'training_history.json'
if history_path.is_file():
    records = json.loads(history_path.read_text(encoding='utf-8'))
    if records:
        frame = pd.json_normalize(records).set_index('epoch')
        frame.index = frame.index + 1
        frame.index.name = 'Эпоха'
        columns = ['train_loss', 'validation_loss', 'validation_source.f1', 'validation_source.eer']
        display(frame[columns].tail())
        frame[columns].plot(subplots=True, layout=(2, 2), figsize=(10, 6), grid=True)
        plt.tight_layout()
        plt.show()
    else:
        print('История пока пуста.')
else:
    print('История появится после первой завершённой эпохи.')